In [ ]:
import os
import glob
import utils
import importlib
import xarray as xr

importlib.reload(utils)

print("starting script", flush=True)
base_path = "../data/ensemble/wind_past"

files_past = []
files_future = []

for root, dirs, files in os.walk(base_path):
    if not dirs:
        rel_path = os.path.relpath(root, base_path)
        path_past = os.path.join(base_path, rel_path)
        future_path = path_past.replace("wind_past", "wind_future")

        # Find nc files
        past_files = glob.glob(os.path.join(path_past, "*1995*.nc"))
        future_files = glob.glob(os.path.join(future_path, "*2045*.nc"))

        files_past.extend(past_files)
        files_future.extend(future_files)


# Use the file names as model identifiers
model_names_past = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_past
]
model_names_future = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_future
]

print("File source loaded", flush=True)
# Process and select time frame
files_raw_past = utils.pre_process(files_past, ["sfcWind"], 0, 5, 0, 5)
files_past = utils.select_time_frame(files_raw_past, slice("1995-01-01", "2004-12-30"))

files_raw_future = utils.pre_process(files_future, ["sfcWind"], 0, 5, 0, 5)
files_future = utils.select_time_frame(
    files_raw_future, slice("2045-01-01", "2054-12-30")
)
print("Files loaded", flush=True)

In [ ]:
results = utils.marginal_average_hourly_diff(
    files_past, files_future, model_names_past, model_names_future
)

In [ ]:
combined_data = xr.concat(results, dim="time")
output_path = "../plotting_data/0069_results.nc"
combined_data.to_netcdf(output_path)

In [ ]:
print("starting script", flush=True)
base_path = "../data/ensemble/rsds_past"

files_past = []
files_future = []

for root, dirs, files in os.walk(base_path):
    if not dirs:  # lowest-level folder
        rel_path = os.path.relpath(root, base_path)
        path_past = os.path.join(base_path, rel_path)
        future_path = path_past.replace("rsds_past", "rsds_future")

        # Find nc files
        past_files = glob.glob(os.path.join(path_past, "*1995*.nc"))
        future_files = glob.glob(os.path.join(future_path, "*2045*.nc"))

        files_past.extend(past_files)
        files_future.extend(future_files)


# Use the file names as model identifiers
model_names_past = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_past
]
model_names_future = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_future
]

print("File source loaded", flush=True)
# Process and select time frame
files_raw_past = utils.pre_process(files_past, ["rsds"], 0, 5, 0, 5)
files_past_rsds = utils.select_time_frame(
    files_raw_past, slice("1995-01-01", "2004-12-30")
)

files_raw_future = utils.pre_process(files_future, ["rsds"], 0, 5, 0, 5)
files_future_rsds = utils.select_time_frame(
    files_raw_future, slice("2045-01-01", "2054-12-30")
)
print("Files loaded", flush=True)

# Make a dictionary mapping models to files
models_past_dict = dict(zip(model_names_past, files_past))
models_future_dict = dict(zip(model_names_future, files_future))

In [ ]:
results_rsds = utils.marginal_average_hourly_diff(
    files_past_rsds, files_future_rsds, model_names_past, model_names_future
)
combined_data = xr.concat(results_rsds, dim="time")
output_path = "../plotting_data/0069_results_rsds.nc"
combined_data.to_netcdf(output_path)